In [ ]:
import os
from google.colab import files

print("Por favor, sube tu archivo kaggle.json")
files.upload() # Aparecerá un botón para subir el archivo

# Configuramos los permisos y el directorio de Kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Descargamos el dataset usando el link exacto que proporcionaste
print("Descargando el dataset de caricaturas...")
!kaggle datasets download -d volkandl/cartoon-classification

# Descomprimimos de forma silenciosa (-q) en la carpeta 'dataset'
print("Descomprimiendo...")
!unzip -q cartoon-classification.zip -d dataset/
print("¡Dataset listo!")

Por favor, sube tu archivo kaggle.json


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, Average, Layer
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
from skimage import color
import matplotlib.pyplot as plt
import os

In [ ]:
class CapaRBFEspacial(Layer):
    def __init__(self, num_centros, gamma=1.0, **kwargs):
        super(CapaRBFEspacial, self).__init__(**kwargs)
        self.num_centros = num_centros
        self.gamma = gamma

    def build(self, input_shape):
        canales = input_shape[-1]
        self.mu = self.add_weight(name='centros_rbf',
                                  shape=(1, 1, 1, canales, self.num_centros),
                                  initializer='random_normal',
                                  trainable=True)
        super(CapaRBFEspacial, self).build(input_shape)

    def call(self, inputs):
        inputs_exp = tf.expand_dims(inputs, axis=-1)
        diferencia = inputs_exp - self.mu
        distancia_l2_cuadrada = tf.reduce_sum(tf.square(diferencia), axis=-2)
        return tf.exp(-self.gamma * distancia_l2_cuadrada)

def construir_red_hibrida_colorizacion(alto=128, ancho=128):
    entrada_L = Input(shape=(alto, ancho, 1), name="Entrada_Escala_Grises")

    # 1. TRONCO COMÚN (Extracción de Contexto con CNN)
    x = Conv2D(64, (3, 3), activation='relu', padding='same', strides=2)(entrada_L)
    x = Conv2D(128, (3, 3), activation='relu', padding='same', strides=2)(x)
    x = Conv2D(256, (3, 3), activation='relu', padding='same')(x)

    x = tf.keras.layers.UpSampling2D((2, 2))(x)
    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = tf.keras.layers.UpSampling2D((2, 2))(x)

    caracteristicas_cnn = Conv2D(64, (3, 3), activation='relu', padding='same')(x)

    # 2. BIFURCACIÓN (Las dos cabezas)
    cabeza_mlp = Conv2D(32, (1, 1), activation='relu')(caracteristicas_cnn)
    salida_mlp = Conv2D(2, (1, 1), activation='tanh', name="Salida_MLP")(cabeza_mlp)

    activacion_rbf = CapaRBFEspacial(num_centros=30, gamma=0.5)(caracteristicas_cnn)
    salida_rbf = Conv2D(2, (1, 1), activation='tanh', name="Salida_RBF")(activacion_rbf)

    # 3. FUSIÓN FINAL
    salida_ensamble = Average(name="Prediccion_Final_ab")([salida_mlp, salida_rbf])

    modelo = Model(inputs=entrada_L, outputs=salida_ensamble)
    modelo.compile(optimizer='adam', loss='mse')

    return modelo

# Instanciamos el modelo
modelo_color = construir_red_hibrida_colorizacion()

In [ ]:
def generador_datos_lab(ruta_dataset, batch_size=16, target_size=(128, 128)):
    datagen = ImageDataGenerator(rescale=1./255)

    # class_mode=None porque las etiquetas de clasificación no nos sirven aquí
    generador = datagen.flow_from_directory(
        ruta_dataset,
        target_size=target_size,
        batch_size=batch_size,
        class_mode=None,
        shuffle=True
    )

    while True:
        try:
            lote_rgb = next(generador)
            # Aseguramos que el lote tenga las dimensiones correctas antes de convertir
            if lote_rgb.shape[0] == 0:
                continue

            lote_lab = color.rgb2lab(lote_rgb)

            x_batch = lote_lab[:, :, :, 0:1]
            x_batch = (x_batch / 50.0) - 1.0 # Normalizar L

            y_batch = lote_lab[:, :, :, 1:3]
            y_batch = y_batch / 128.0        # Normalizar a, b

            yield (x_batch, y_batch)
        except Exception as e:
            # En caso de alguna imagen corrupta en el dataset, la ignora y sigue
            continue

In [ ]:
# El dataset de Kaggle suele crear carpetas TRAIN y TEST al descomprimirse
ruta_train = 'dataset/cartoon_classification/TRAIN'
ruta_test = 'dataset/cartoon_classification/TEST'

# Iniciamos los generadores
gen_entrenamiento = generador_datos_lab(ruta_train, batch_size=16)
gen_prueba = generador_datos_lab(ruta_test, batch_size=1)

print("Iniciando entrenamiento del ensamble CNN + MLP/RBF...")
# Usamos 200 pasos por época para que no tarde una eternidad,
# pero puedes ajustarlo según lo que necesites para tu demostración.
historia = modelo_color.fit(
    gen_entrenamiento,
    steps_per_epoch=200,
    epochs=15
)

In [ ]:
# 1. Extraemos una imagen de prueba
L_real_norm, ab_real_norm = next(gen_prueba)

# 2. Hacemos la predicción
print("La red está coloreando la imagen...")
ab_predicho_norm = modelo_color.predict(L_real_norm)

# 3. Desnormalizamos todos los tensores
L = (L_real_norm[0] + 1.0) * 50.0
ab_predicho = ab_predicho_norm[0] * 128.0
ab_real = ab_real_norm[0] * 128.0

# 4. Unimos los canales
imagen_predicha_lab = np.concatenate([L, ab_predicho], axis=-1)
imagen_real_lab = np.concatenate([L, ab_real], axis=-1)

# 5. Convertimos a RGB
# Usamos np.clip para evitar advertencias si algún pixel se sale del límite al desnormalizar
imagen_predicha_rgb = np.clip(color.lab2rgb(imagen_predicha_lab), 0, 1)
imagen_real_rgb = np.clip(color.lab2rgb(imagen_real_lab), 0, 1)

# 6. Graficamos los resultados comparativos
plt.figure(figsize=(15, 5))

# Imagen 1: Entrada (Escala de grises)
plt.subplot(1, 3, 1)
plt.imshow(L[:, :, 0], cmap='gray')
plt.title('Entrada (Canal L)')
plt.axis('off')

# Imagen 2: Predicción de tu Red
plt.subplot(1, 3, 2)
plt.imshow(imagen_predicha_rgb)
plt.title('Coloreado por IA (Ensamble)')
plt.axis('off')

# Imagen 3: Ground Truth (Real)
plt.subplot(1, 3, 3)
plt.imshow(imagen_real_rgb)
plt.title('Color Original Real')
plt.axis('off')

plt.tight_layout()
plt.show()